# IOAI — 2025 National Selection Duplicate Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, json, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-national-selection-duplicate-detection/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
# BERT(115MB)는 github 한계로 bohrium 공개 API 의 신선한 서명 URL 로 받는다
if not os.path.isdir('data/my_custom_bert_model'):
    d = json.load(urllib.request.urlopen('https://www.bohrium.com/bohrapi/v1/square/competition/7927749324'))
    uri = [x for x in d['data']['datasetList'] if x['dsName'].endswith('train')][0]['downloadUri']
    urllib.request.urlretrieve(uri, 'train.zip')
    with zipfile.ZipFile('train.zip') as z:
        for n in z.namelist():
            if n.startswith('my_custom_bert_model/'): z.extract(n, 'data')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Duplicate Detection — 질문의 의미적 동등성 (Georgia National Selection 2025)

제공된 소형 **BERT**(`data/my_custom_bert_model/`)를 파인튜닝해 질문쌍이 의미상 **동등(1)** 한지 판별한다.
학습 데이터는 특이하다: `train_data.csv`(동등쌍 400개, 전부 label 1) + `raw_questions.csv`(서로 무관한 문장 1000개).
음성쌍은 raw 문장을 조합해 **직접 만들어야** 한다. 채점: held-out **F1**. 제출: `submission.json`={idx: 0/1}. T4 ≤1h.

In [ ]:
import pandas as pd, json
train = pd.read_csv("data/train_data.csv")       # 동등쌍(400), is_duplicate=1
raw   = pd.read_csv("data/raw_questions.csv")     # 무관 문장 1000
test  = pd.read_csv("data/test.csv")              # 예측 대상 쌍(question1,question2)
print("pos pairs", len(train), "| raw", len(raw), "| test", len(test))

In [ ]:
# 베이스라인: 전부 0 (starter) — F1 = 0
predictions = {str(i): 0 for i in range(len(test))}
with open("submission.json", "w") as f:
    json.dump(predictions, f, indent=4)
print("saved submission.json", len(predictions))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)